# Práctica integradora — NYC Taxi Analytics (Databricks)

**Materia:** Tópicos de Big Data · **Periodo:** enero–marzo de 2025 · **Tecnologías:** Databricks Free Edition, PySpark, Spark SQL, Delta Lake, Unity Catalog y Tableau Public.

## Propósito
Construir un flujo completo de datos: **ingesta → diagnóstico → limpieza → integración → tablas Delta → indicadores → visualización**. Trabajaremos con millones de viajes sin descargarlos a la computadora local: los archivos se obtienen desde el entorno de Databricks y se guardan en un volumen de Unity Catalog.

### Instrucciones para el estudiante
1. Importa este `.ipynb` en **Workspace → Import** de Databricks y conéctalo a cómputo compatible.
2. Ejecuta las celdas **en orden**; lee la explicación que precede a cada celda y responde las preguntas de reflexión.
3. Se requieren permisos para crear catálogo, esquemas, volumen y tablas. Si el catálogo compartido ya existe, evita sobrescribir trabajos de otros equipos: utiliza un catálogo o esquemas propios.
4. La descarga requiere acceso HTTPS desde el cómputo de Databricks. Las operaciones de conteo, agrupación y escritura pueden tardar varios minutos.
5. **No ejecutes indiscriminadamente sobre tablas ajenas:** `overwrite` y `CREATE OR REPLACE TABLE` reemplazan resultados existentes.

**Fuentes oficiales:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) y el catálogo `taxi_zone_lookup.csv` de NYC TLC.

**Criterio analítico:** conservaremos viajes de enero a marzo de 2025 con distancia y duración positivas. No eliminaremos automáticamente tarifas negativas ni valores atípicos; su tratamiento depende del indicador. Los conteos de referencia pueden cambiar si se actualizan los archivos de origen.

## Etapa 1 · Ingesta y preparación

Define la URL del archivo Parquet de enero de 2025. Se usará para verificar conectividad antes de descargar datos.

In [0]:
# NYC Taxi Analytics
# # Validacion de acceso a fuentes ENERO

# Etapa 1: Ingesta de datos

url_enero = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2025-01.parquet"
)

print(url_enero)

https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet


Hace una solicitud HTTP parcial y muestra el código de respuesta y el tipo de contenido. Una respuesta exitosa indica que el origen es accesible; esta comprobación no garantiza la descarga completa.

In [0]:
import requests

respuesta = requests.get(
    url_enero,
    headers={"Range": "bytes=0-15"},
    stream=True,
    timeout=30
)

print("Código HTTP:", respuesta.status_code)
print("Tipo de contenido:",
      respuesta.headers.get("Content-Type"))

respuesta.close()

Código HTTP: 206
Tipo de contenido: binary/octet-stream


Identifica el catálogo y esquema activos de Unity Catalog. Permite reconocer el contexto de ejecución antes de crear objetos.

In [0]:
print("Catálogo actual:")
spark.sql("SELECT current_catalog()").show()

print("Esquema actual:")
spark.sql("SELECT current_schema()").show()

print("Catálogos disponibles:")
spark.sql("SHOW CATALOGS").show()

Catálogo actual:
+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+

Esquema actual:
+----------------+
|current_schema()|
+----------------+
|         default|
+----------------+

Catálogos disponibles:
+---------+
|  catalog|
+---------+
|  samples|
|   system|
|workspace|
+---------+



Crea el catálogo del proyecto si no existe. Requiere permisos de Unity Catalog; no modifica catálogos existentes.

In [0]:
spark.sql("""
    CREATE CATALOG IF NOT EXISTS nyc_taxi_analytics;
""")

DataFrame[]

Crea tres esquemas: `stage` para archivos de origen, `clean` para datos depurados y `datamart` para indicadores agregados.

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS nyc_taxi_analytics.stage
""")

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS nyc_taxi_analytics.clean
""")

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS nyc_taxi_analytics.datamart
""")

DataFrame[]

Comprueba que los esquemas estén disponibles. **Verifica:** aparecen `stage`, `clean` y `datamart`.

In [0]:
spark.sql("""
    SHOW SCHEMAS IN nyc_taxi_analytics
""").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|clean             |
|datamart          |
|default           |
|information_schema|
|stage             |
+------------------+



Crea un volumen administrado en `stage`. Un volumen permite almacenar archivos de datos accesibles mediante `/Volumes/...`.

In [0]:
spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    nyc_taxi_analytics.stage.archivos_origen
""")

DataFrame[]

Confirma que el volumen `archivos_origen` exista.

In [0]:
spark.sql("""
    SHOW VOLUMES IN nyc_taxi_analytics.stage
""").show(truncate=False)

+--------+---------------+
|database|volume_name    |
+--------+---------------+
|stage   |archivos_origen|
+--------+---------------+



Define rutas de origen y destino para enero. El destino está en almacenamiento de Databricks, **no en tu laptop**.

In [0]:
url_enero = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2025-01.parquet"
)

ruta_volumen = (
    "/Volumes/nyc_taxi_analytics/stage/"
    "archivos_origen/"
)

archivo_enero = ruta_volumen + "yellow_tripdata_2025-01.parquet"

print("Origen:", url_enero)
print("Destino:", archivo_enero)

Origen: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet
Destino: /Volumes/nyc_taxi_analytics/stage/archivos_origen/yellow_tripdata_2025-01.parquet


Descarga enero en bloques para evitar cargar todo el archivo en memoria. Se ejecuta en Databricks; si el archivo ya existe, volverá a descargarse.

In [0]:
import requests

with requests.get(url_enero, stream=True, timeout=120) as respuesta:
    respuesta.raise_for_status()

    with open(archivo_enero, "wb") as archivo:
        for bloque in respuesta.iter_content(chunk_size=1024 * 1024):
            if bloque:
                archivo.write(bloque)

print("Archivo transferido correctamente.")

Archivo transferido correctamente.


Verifica el tamaño del archivo descargado. Un tamaño mayor que cero es una comprobación preliminar, no una validación exhaustiva.

In [0]:
import os

tamano_mb = os.path.getsize(archivo_enero) / (1024 ** 2)

print(f"Tamaño del archivo: {tamano_mb:.2f} MB")

Tamaño del archivo: 56.42 MB


Lee enero como DataFrame distribuido de Spark. Spark interpreta el esquema Parquet.

In [0]:
ruta_enero = (
    "/Volumes/nyc_taxi_analytics/stage/"
    "archivos_origen/yellow_tripdata_2025-01.parquet"
)

df_enero = spark.read.parquet(ruta_enero)

print("Archivo Parquet leído correctamente.")

Archivo Parquet leído correctamente.


Muestra nombres y tipos de las columnas. Identifica fechas, distancias, tarifas e identificadores de zona.

In [0]:
df_enero.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



Inspecciona una muestra pequeña. `limit` evita traer todos los registros a la interfaz.

In [0]:
display(df_enero.limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0
2,2025-01-01T00:48:24.000,2025-01-01T01:08:26.000,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0
2,2025-01-01T00:00:02.000,2025-01-01T00:09:36.000,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1.0,16.4,2.5,0.0,0.0


Cuenta filas y columnas del archivo de enero. `count()` activa un trabajo distribuido.

In [0]:
num_filas = df_enero.count()
num_columnas = len(df_enero.columns)

print(f"Número de registros: {num_filas:,}")
print(f"Número de columnas: {num_columnas}")

Número de registros: 3,475,226
Número de columnas: 20


Descarga febrero y marzo al mismo volumen mediante un ciclo. Cada archivo se transfiere en bloques y se verifica su tamaño.

In [0]:
import requests
import os

meses = ["02", "03"]

url_base = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/"
)

ruta_volumen = (
    "/Volumes/nyc_taxi_analytics/stage/"
    "archivos_origen/"
)

for mes in meses:

    nombre_archivo = f"yellow_tripdata_2025-{mes}.parquet"

    url = url_base + nombre_archivo
    destino = ruta_volumen + nombre_archivo

    print(f"Descargando {nombre_archivo}...")

    with requests.get(url, stream=True, timeout=120) as respuesta:
        respuesta.raise_for_status()

        with open(destino, "wb") as archivo:
            for bloque in respuesta.iter_content(chunk_size=1024 * 1024):
                if bloque:
                    archivo.write(bloque)

    tamano_mb = os.path.getsize(destino) / (1024 ** 2)

    print(f"Archivo guardado: {tamano_mb:.2f} MB\n")

Descargando yellow_tripdata_2025-02.parquet...
Archivo guardado: 57.55 MB

Descargando yellow_tripdata_2025-03.parquet...
Archivo guardado: 66.72 MB



Verifica que existan los tres archivos mensuales y muestra sus tamaños.

In [0]:
for mes in ["01", "02", "03"]:

    archivo = (
        ruta_volumen +
        f"yellow_tripdata_2025-{mes}.parquet"
    )

    tamano_mb = os.path.getsize(archivo) / (1024 ** 2)

    print(f"2025-{mes}: {tamano_mb:.2f} MB")

2025-01: 56.42 MB
2025-02: 57.55 MB
2025-03: 66.72 MB


## Etapa 2 · Lectura y exploración de los viajes

Lee los tres Parquet con un comodín (`*`) y construye `df_viajes_3meses`. No concatena los datos en memoria local.

In [0]:
ruta_viajes_3meses = (
    "/Volumes/nyc_taxi_analytics/stage/"
    "archivos_origen/yellow_tripdata_2025-*.parquet"
)

df_viajes_3meses = spark.read.parquet(ruta_viajes_3meses)

print("DataFrame trimestral creado correctamente.")

DataFrame trimestral creado correctamente.


Cuenta el total de registros y columnas del trimestre. **Referencia:** 11,198,026 filas y 20 columnas.

In [0]:
num_filas = df_viajes_3meses.count()
num_columnas = len(df_viajes_3meses.columns)

print(f"Total de viajes: {num_filas:,}")
print(f"Total de columnas: {num_columnas}")

df_viajes_3meses.printSchema()

Total de viajes: 11,198,026
Total de columnas: 20
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



Agrupa por mes de la fecha real de inicio. Detecta registros fechados fuera del trimestre aunque procedan de los archivos seleccionados.

In [0]:
from pyspark.sql import functions as F

df_viajes_3meses.groupBy(
    F.date_format("tpep_pickup_datetime", "yyyy-MM").alias("mes")
).count().orderBy("mes").show()

+-------+-------+
|    mes|  count|
+-------+-------+
|2007-12|      1|
|2009-01|      1|
|2024-12|     21|
|2025-01|3475234|
|2025-02|3577542|
|2025-03|4145225|
|2025-04|      2|
+-------+-------+



## Etapa 3 · Catálogo de zonas

Define URL y ruta del CSV oficial que relaciona identificadores de zona con nombres y distritos.

In [0]:
# Incorporar el catálogo de zonas de taxi_zone_lookup.csv

url_zonas = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "misc/taxi_zone_lookup.csv"
)

ruta_zonas = (
    "/Volumes/nyc_taxi_analytics/stage/"
    "archivos_origen/taxi_zone_lookup.csv"
)

Descarga el catálogo de zonas al volumen y comprueba su tamaño.

In [0]:
import requests
import os

with requests.get(url_zonas, stream=True, timeout=60) as respuesta:
    respuesta.raise_for_status()

    with open(ruta_zonas, "wb") as archivo:
        for bloque in respuesta.iter_content(chunk_size=1024 * 1024):
            if bloque:
                archivo.write(bloque)

print("Archivo descargado correctamente.")
print(f"Tamaño: {os.path.getsize(ruta_zonas) / 1024:.2f} KB")

Archivo descargado correctamente.
Tamaño: 12.04 KB


Lee el CSV como DataFrame; `header` interpreta encabezados y `inferSchema` deduce tipos. En producción conviene declarar un esquema explícito.

In [0]:
df_zonas = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_zonas)
)

df_zonas.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



Inspecciona filas y dimensiones del catálogo. **Referencia:** 265 zonas.

In [0]:
display(df_zonas.limit(10))

print("Total de zonas:", df_zonas.count())
print("Total de columnas:", len(df_zonas.columns))

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


Total de zonas: 265
Total de columnas: 4


Verifica unicidad y valores nulos de `LocationID`. Un catálogo con claves repetidas podría multiplicar filas al hacer un JOIN.

In [0]:
from pyspark.sql import functions as F

df_zonas.select(
    F.count("*").alias("total_registros"),
    F.countDistinct("LocationID").alias("zonas_distintas"),
    F.sum(F.col("LocationID").isNull().cast("int")).alias("ids_nulos")
).show()

+---------------+---------------+---------+
|total_registros|zonas_distintas|ids_nulos|
+---------------+---------------+---------+
|            265|            265|        0|
+---------------+---------------+---------+



Registra una vista temporal llamada `zonas` para consultarla con Spark SQL.

In [0]:
df_zonas.createOrReplaceTempView("zonas")

Repite la validación del catálogo mediante SQL. Sirve para comparar el enfoque SQL con la API DataFrame.

In [0]:
spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT LocationID) AS ids_distintos,
        SUM(CASE WHEN LocationID IS NULL THEN 1 ELSE 0 END) AS ids_nulos
    FROM zonas
""").show()

+---------------+-------------+---------+
|total_registros|ids_distintos|ids_nulos|
+---------------+-------------+---------+
|            265|          265|        0|
+---------------+-------------+---------+



## Etapa 4 · Diagnóstico de calidad

Registra `viajes` como vista temporal. Las consultas posteriores usarán SQL sobre este DataFrame.

In [0]:
# Calidad de los datos

#df_viajes_3meses.show(3)
df_viajes_3meses.createOrReplaceTempView("viajes")

Cuenta valores nulos en columnas relevantes. Observa especialmente `passenger_count`: su ausencia no obliga a descartar el viaje.

In [0]:
# valores nulos
spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(*) - COUNT(tpep_pickup_datetime) AS nulos_fecha_inicio,
        COUNT(*) - COUNT(tpep_dropoff_datetime) AS nulos_fecha_fin,
        COUNT(*) - COUNT(PULocationID) AS nulos_zona_origen,
        COUNT(*) - COUNT(DOLocationID) AS nulos_zona_destino,
        COUNT(*) - COUNT(trip_distance) AS nulos_distancia,
        COUNT(*) - COUNT(fare_amount) AS nulos_tarifa,
        COUNT(*) - COUNT(passenger_count) AS nulos_pasajeros
    FROM viajes
""").show()

+---------------+------------------+---------------+-----------------+------------------+---------------+------------+---------------+
|total_registros|nulos_fecha_inicio|nulos_fecha_fin|nulos_zona_origen|nulos_zona_destino|nulos_distancia|nulos_tarifa|nulos_pasajeros|
+---------------+------------------+---------------+-----------------+------------------+---------------+------------+---------------+
|       11198026|                 0|              0|                0|                 0|              0|           0|        2263749|
+---------------+------------------+---------------+-----------------+------------------+---------------+------------+---------------+



Cuenta distancias, tarifas y duraciones problemáticas. Estas condiciones pueden superponerse, por lo que no se deben sumar como si fueran filas diferentes.

In [0]:
# valores inconsistentes
spark.sql("""
    SELECT
        COUNT(*) AS total_registros,

        SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END)
            AS distancias_negativas,

        SUM(CASE WHEN trip_distance = 0 THEN 1 ELSE 0 END)
            AS distancias_cero,

        SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END)
            AS tarifas_negativas,

        SUM(CASE WHEN fare_amount = 0 THEN 1 ELSE 0 END)
            AS tarifas_cero,

        SUM(CASE WHEN tpep_dropoff_datetime < tpep_pickup_datetime
                 THEN 1 ELSE 0 END)
            AS duraciones_negativas,

        SUM(CASE WHEN tpep_dropoff_datetime = tpep_pickup_datetime
                 THEN 1 ELSE 0 END)
            AS duraciones_cero

    FROM viajes
""").show(truncate=False)

+---------------+--------------------+---------------+-----------------+------------+--------------------+---------------+
|total_registros|distancias_negativas|distancias_cero|tarifas_negativas|tarifas_cero|duraciones_negativas|duraciones_cero|
+---------------+--------------------+---------------+-----------------+------------+--------------------+---------------+
|11198026       |0                   |294386         |535497           |4493        |298                 |29197          |
+---------------+--------------------+---------------+-----------------+------------+--------------------+---------------+



Interpreta las anomalías detectadas. Diferencia un registro incorrecto de una operación especial que merece investigación.

Hay tres hallazgos importantes:
- 294,386 viajes con distancia cero. Debemos revisar si son registros especiales, viajes cancelados o errores de captura.
- 535,497 tarifas negativas. Es una proporción suficientemente alta como para investigar su relación con el tipo de pago y otros campos.
- 298 duraciones negativas. Son inconsistencias temporales claras que deberíamos excluir del análisis de duración.

Relaciona tarifas negativas con el tipo de pago. El objetivo es entender el patrón antes de definir filtros.

In [0]:
# Investigar, antes de limpiar

spark.sql("""
    SELECT
        payment_type,

        COUNT(*) AS total_viajes,

        SUM(
            CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END
        ) AS tarifas_negativas,

        ROUND(
            100.0 * SUM(
                CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END
            ) / COUNT(*),
            2
        ) AS porcentaje_negativas

    FROM viajes
    GROUP BY payment_type
    ORDER BY tarifas_negativas DESC
""").show(truncate=False)

+------------+------------+-----------------+--------------------+
|payment_type|total_viajes|tarifas_negativas|porcentaje_negativas|
+------------+------------+-----------------+--------------------+
|0           |2263749     |353160           |15.60               |
|4           |241239      |115141           |47.73               |
|2           |1135757     |43358            |3.82                |
|3           |72546       |23791            |32.79               |
|1           |7484734     |47               |0.00                |
|5           |1           |0                |0.00                |
+------------+------------+-----------------+--------------------+



Distingue calidad de datos y selección analítica: una tarifa no positiva puede excluirse del promedio de tarifas sin eliminarse de la demanda de movilidad.

* IMPORTANTE: Distinguir entre calidad de datos y criterios de selección analítica.

Presenta la interpretación de códigos de pago de la fuente. Contrasta los códigos con el diccionario de datos de NYC TLC.

Según el diccionario de datos de NYC TLC, los códigos habituales de payment_type son:

Código	Significado

0	No especificado o no registrado.<br>
1	Tarjeta de crédito.<br>
2	Efectivo.<br>
3	Sin cargo.<br>
4	Disputa.<br>
5	Desconocido.<br>

La interpretación es interesante: las tarifas negativas se concentran en pagos no especificados, disputas y transacciones sin cargo.

En particular, casi el 48 % de los registros con payment_type = 4 tienen tarifas negativas.

Esto sugiere que parte de esos valores podrían estar relacionados con ajustes o disputas de cobro. Sin embargo, no podemos asegurar que todos correspondan a transacciones administrativas únicamente por el código de pago.

Analiza conjuntamente `payment_type` y la ausencia de `passenger_count`. Permite detectar valores faltantes sistemáticos.

In [0]:
spark.sql("""
    SELECT
        payment_type,
        COUNT(*) AS total_registros,

        SUM(
            CASE WHEN passenger_count IS NULL
            THEN 1 ELSE 0 END
        ) AS pasajeros_nulos,

        ROUND(
            100.0 * SUM(
                CASE WHEN passenger_count IS NULL
                THEN 1 ELSE 0 END
            ) / COUNT(*),
            2
        ) AS porcentaje_nulos

    FROM viajes
    GROUP BY payment_type
    ORDER BY payment_type
""").show(truncate=False)

+------------+---------------+---------------+----------------+
|payment_type|total_registros|pasajeros_nulos|porcentaje_nulos|
+------------+---------------+---------------+----------------+
|0           |2263749        |2263749        |100.00          |
|1           |7484734        |0              |0.00            |
|2           |1135757        |0              |0.00            |
|3           |72546          |0              |0.00            |
|4           |241239         |0              |0.00            |
|5           |1              |0              |0.00            |
+------------+---------------+---------------+----------------+



Documenta el hallazgo de que los registros con tipo de pago 0 presentan pasajeros nulos; no imputar cero sin justificación.

Interpretación del resultado
Los 2,263,749 registros con payment_type = 0 tienen passenger_count nulo. Además, ninguno de los registros con otros tipos de pago presenta ese problema.

Esto significa que, en nuestros datos:
- El 20.22 % de los viajes tiene simultáneamente payment_type = 0 y passenger_count = NULL.
- La ausencia de pasajeros no está distribuida aleatoriamente entre los tipos de pago.
- Ambos campos parecen estar relacionados con una misma característica del proceso de generación de los registros.
No podemos afirmar todavía cuál es la causa, pero sí podemos demostrar la asociación.

Este hallazgo es muy útil para explicar a los alumnos que la calidad de datos no consiste únicamente en contar valores nulos, sino también en investigar sus patrones.

Busca filas exactamente iguales en todas las columnas. `COALESCE` devuelve cero cuando no hay grupos repetidos. **Advertencia:** agrupar millones de filas por todas las columnas puede ser costoso.

In [0]:
# Continuar con el diagnóstico: registros duplicados

columnas = ", ".join(
    f"`{columna}`"
    for columna in df_viajes_3meses.columns
)

consulta_duplicados = f"""
    SELECT
        COALESCE(SUM(repeticiones - 1), 0) AS registros_duplicados
    FROM (
        SELECT
            COUNT(*) AS repeticiones
        FROM viajes
        GROUP BY {columnas}
        HAVING COUNT(*) > 1
    )
"""

spark.sql(consulta_duplicados).show()

+--------------------+
|registros_duplicados|
+--------------------+
|                   0|
+--------------------+



Calcula mínimo, media y máximo de distancia y tarifa. Los máximos extraordinarios muestran por qué la media puede ser sensible a atípicos.

In [0]:
# estadistica descriptiva

spark.sql("""
    SELECT
        ROUND(MIN(trip_distance), 2) AS distancia_min,
        ROUND(AVG(trip_distance), 2) AS distancia_promedio,
        ROUND(MAX(trip_distance), 2) AS distancia_max,

        ROUND(MIN(fare_amount), 2) AS tarifa_min,
        ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
        ROUND(MAX(fare_amount), 2) AS tarifa_max

    FROM viajes
""").show()

+-------------+------------------+-------------+----------+---------------+----------+
|distancia_min|distancia_promedio|distancia_max|tarifa_min|tarifa_promedio|tarifa_max|
+-------------+------------------+-------------+----------+---------------+----------+
|          0.0|              6.18|    320136.29|   -1807.6|          17.24| 863372.12|
+-------------+------------------+-------------+----------+---------------+----------+



Calcula percentiles aproximados P50, P95 y P99, adecuados para grandes volúmenes. Los percentiles **no son reglas automáticas de eliminación**.

In [0]:
# analisis de percientiles

spark.sql("""
    SELECT
        percentile_approx(trip_distance, 0.50) AS distancia_p50,
        percentile_approx(trip_distance, 0.95) AS distancia_p95,
        percentile_approx(trip_distance, 0.99) AS distancia_p99,

        percentile_approx(fare_amount, 0.50) AS tarifa_p50,
        percentile_approx(fare_amount, 0.95) AS tarifa_p95,
        percentile_approx(fare_amount, 0.99) AS tarifa_p99

    FROM viajes
""").show()

+-------------+-------------+-------------+----------+----------+----------+
|distancia_p50|distancia_p95|distancia_p99|tarifa_p50|tarifa_p95|tarifa_p99|
+-------------+-------------+-------------+----------+----------+----------+
|         1.72|        11.64|        19.33|      12.8|      51.3|      71.6|
+-------------+-------------+-------------+----------+----------+----------+



Calcula estadísticas de duración usando las dos marcas de tiempo. Duraciones negativas o enormes requieren diagnóstico.

In [0]:
# Analizar la duración de los viajes

spark.sql("""
    WITH duraciones AS (
        SELECT
            (
                timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime)
            ) / 60.0 AS duracion_minutos
        FROM viajes
    )

    SELECT
        ROUND(MIN(duracion_minutos), 2) AS duracion_min,
        ROUND(AVG(duracion_minutos), 2) AS duracion_promedio,
        ROUND(MAX(duracion_minutos), 2) AS duracion_max,

        ROUND(percentile_approx(duracion_minutos, 0.50), 2) AS duracion_p50,
        ROUND(percentile_approx(duracion_minutos, 0.95), 2) AS duracion_p95,
        ROUND(percentile_approx(duracion_minutos, 0.99), 2) AS duracion_p99

    FROM duraciones
""").show()

+------------+-----------------+------------+------------+------------+------------+
|duracion_min|duracion_promedio|duracion_max|duracion_p50|duracion_p95|duracion_p99|
+------------+-----------------+------------+------------+------------+------------+
|   -51472.32|            15.50|     8564.67|       12.13|       37.18|       61.43|
+------------+-----------------+------------+------------+------------+------------+



Clasifica viajes en intervalos de duración. Ayuda a cuantificar cuántos registros están en los extremos.

In [0]:
# Distribución de viajes por intervalos de duración

spark.sql("""
    WITH viajes_duracion AS (
        SELECT
            timestampdiff(
                SECOND,
                tpep_pickup_datetime,
                tpep_dropoff_datetime
            ) / 60.0 AS duracion_minutos
        FROM viajes
    )

    SELECT
        CASE
            WHEN duracion_minutos <= 0 THEN '01. No positiva'
            WHEN duracion_minutos <= 30 THEN '02. Hasta 30 min'
            WHEN duracion_minutos <= 60 THEN '03. 30 a 60 min'
            WHEN duracion_minutos <= 120 THEN '04. 1 a 2 horas'
            WHEN duracion_minutos <= 240 THEN '05. 2 a 4 horas'
            ELSE '06. Más de 4 horas'
        END AS intervalo_duracion,

        COUNT(*) AS total_viajes

    FROM viajes_duracion
    GROUP BY intervalo_duracion
    ORDER BY intervalo_duracion
""").show(truncate=False)

+------------------+------------+
|intervalo_duracion|total_viajes|
+------------------+------------+
|01. No positiva   |29495       |
|02. Hasta 30 min  |10205232    |
|03. 30 a 60 min   |839095      |
|04. 1 a 2 horas   |117477      |
|05. 2 a 4 horas   |2765        |
|06. Más de 4 horas|3962        |
+------------------+------------+



Define reglas básicas de validez y explica por qué se evitan umbrales superiores arbitrarios.

## Establecer nuestras reglas de limpieza

La propuesta es distinguir entre reglas de validez y límites para detectar valores atípicos.

| Variable|Regla| Tratamiento|
|---|---|---|
|Validez temporal|Inicio dentro de Enero–marzo de 2025|Excluir registros fuera del periodo|
|Duración|Mayor que 0 minutos|Excluir duraciones no positivas o nulas|
|Distancia|Mayor que 0 millas|Excluir distancias nulas o sin desplazamiento|
|Tarifa|Mayor que 0 USD|Excluir del analisis de tarifas positivas de viajes|
|Pasajeros|Conservar nulos|Conservar información ya que no son indispensables para nuestros indicadores|
|Zonas|Conservar identificadores para validar contra catálogo|Evitar perder viajes antes del JOIN|
|Valores extremos|Investigar límites superiores|No eliminar automáticamente|

Cuenta viajes que superan distintos umbrales exploratorios de distancia, tarifa y duración. Las categorías se superponen.

In [0]:
# Cuantificar los valores extremos

spark.sql("""
    WITH viajes_metricas AS (
        SELECT
            trip_distance,
            fare_amount,
            timestampdiff(
                SECOND,
                tpep_pickup_datetime,
                tpep_dropoff_datetime
            ) / 60.0 AS duracion_minutos
        FROM viajes
    )

    SELECT
        COUNT(*) AS total_viajes,

        SUM(CASE WHEN trip_distance > 20 THEN 1 ELSE 0 END)
            AS distancia_mayor_20,

        SUM(CASE WHEN trip_distance > 50 THEN 1 ELSE 0 END)
            AS distancia_mayor_50,

        SUM(CASE WHEN trip_distance > 100 THEN 1 ELSE 0 END)
            AS distancia_mayor_100,

        SUM(CASE WHEN fare_amount > 100 THEN 1 ELSE 0 END)
            AS tarifa_mayor_100,

        SUM(CASE WHEN fare_amount > 500 THEN 1 ELSE 0 END)
            AS tarifa_mayor_500,

        SUM(CASE WHEN duracion_minutos > 120 THEN 1 ELSE 0 END)
            AS duracion_mayor_120,

        SUM(CASE WHEN duracion_minutos > 240 THEN 1 ELSE 0 END)
            AS duracion_mayor_240

    FROM viajes_metricas
""").show(truncate=False)

+------------+------------------+------------------+-------------------+----------------+----------------+------------------+------------------+
|total_viajes|distancia_mayor_20|distancia_mayor_50|distancia_mayor_100|tarifa_mayor_100|tarifa_mayor_500|duracion_mayor_120|duracion_mayor_240|
+------------+------------------+------------------+-------------------+----------------+----------------+------------------+------------------+
|11198026    |84817             |1581              |548                |24658           |146             |6727              |3962              |
+------------+------------------+------------------+-------------------+----------------+----------------+------------------+------------------+



Calcula cuántas filas conservaríamos con las reglas de movilidad y cuántas servirían para analizar tarifas positivas. **Referencia:** 10,876,240 y 10,385,969.

In [0]:
# Analizar reglas

spark.sql("""
    WITH viajes_evaluados AS (
        SELECT
            *,
            timestampdiff(
                SECOND,
                tpep_pickup_datetime,
                tpep_dropoff_datetime
            ) / 60.0 AS duracion_minutos
        FROM viajes
    )

    SELECT
        COUNT(*) AS registros_originales,

        SUM(
            CASE
                WHEN tpep_pickup_datetime >= '2025-01-01'
                 AND tpep_pickup_datetime < '2025-04-01'
                 AND duracion_minutos > 0
                 AND trip_distance > 0
                THEN 1 ELSE 0
            END
        ) AS registros_validos_movilidad,

        SUM(
            CASE
                WHEN tpep_pickup_datetime >= '2025-01-01'
                 AND tpep_pickup_datetime < '2025-04-01'
                 AND duracion_minutos > 0
                 AND trip_distance > 0
                 AND fare_amount > 0
                THEN 1 ELSE 0
            END
        ) AS registros_validos_tarifas

    FROM viajes_evaluados
""").show(truncate=False)

+--------------------+---------------------------+-------------------------+
|registros_originales|registros_validos_movilidad|registros_validos_tarifas|
+--------------------+---------------------------+-------------------------+
|11198026            |10876240                   |10385969                 |
+--------------------+---------------------------+-------------------------+



Comienza la transformación distribuida con PySpark a partir del diagnóstico anterior.

## Transformación con PySpark
Crear el DataFrame de viajes limpios

## Etapa 5 · Limpieza con PySpark

Calcula `duracion_minutos`, filtra fechas del trimestre y valores positivos de distancia/duración; deriva fecha, mes, día de semana y hora. `dayofweek`: domingo=1, sábado=7.

In [0]:
from pyspark.sql import functions as F

df_viajes_clean = (
    df_viajes_3meses

    # Calcular duración del viaje en minutos
    .withColumn(
        "duracion_minutos",
        F.timestamp_diff(
            "SECOND",
            F.col("tpep_pickup_datetime"),
            F.col("tpep_dropoff_datetime")
        ) / 60.0
    )

    # Aplicar reglas básicas de validez
    .filter(
        (F.col("tpep_pickup_datetime") >= "2025-01-01") &
        (F.col("tpep_pickup_datetime") < "2025-04-01") &
        (F.col("duracion_minutos") > 0) &
        (F.col("trip_distance") > 0)
    )

    # Incorporar variables para el análisis
    .withColumn(
        "fecha_viaje",
        F.to_date("tpep_pickup_datetime")
    )
    .withColumn(
        "mes",
        F.month("tpep_pickup_datetime")
    )
    .withColumn(
        "dia_semana",
        F.dayofweek("tpep_pickup_datetime")
    )
    .withColumn(
        "hora",
        F.hour("tpep_pickup_datetime")
    )
)

Verifica el conteo y muestra ejemplos. **Referencia:** 10,876,240 viajes; si no coincide, revisa origen y filtros.

In [0]:
print("Registros limpios:", df_viajes_clean.count())

df_viajes_clean.select(
    "fecha_viaje",
    "mes",
    "dia_semana",
    "hora",
    "trip_distance",
    "duracion_minutos",
    "fare_amount"
).show(5, truncate=False)

Registros limpios: 10876240
+-----------+---+----------+----+-------------+------------------+-----------+
|fecha_viaje|mes|dia_semana|hora|trip_distance|duracion_minutos  |fare_amount|
+-----------+---+----------+----+-------------+------------------+-----------+
|2025-03-01 |3  |7         |0   |0.9          |8.6               |7.9        |
|2025-03-01 |3  |7         |0   |0.6          |6.216666666666667 |6.5        |
|2025-03-01 |3  |7         |0   |1.94         |15.233333333333333|14.9       |
|2025-03-01 |3  |7         |0   |0.95         |5.316666666666666 |7.2        |
|2025-03-01 |3  |7         |0   |1.5          |8.266666666666667 |8.6        |
+-----------+---+----------+----+-------------+------------------+-----------+
only showing top 5 rows


## Etapa 6 · Integración geográfica

Realiza dos LEFT JOIN con el catálogo: uno para origen y otro para destino. Los alias evitan confundir las columnas de cada relación.

In [0]:
from pyspark.sql import functions as F

df_viajes_integrados = (
    df_viajes_clean.alias("v")

    # Zona de origen
    .join(
        df_zonas.alias("origen"),
        F.col("v.PULocationID") == F.col("origen.LocationID"),
        "left"
    )

    # Zona de destino
    .join(
        df_zonas.alias("destino"),
        F.col("v.DOLocationID") == F.col("destino.LocationID"),
        "left"
    )

    # Conservar columnas de viajes y agregar datos geográficos
    .select(
        "v.*",

        F.col("origen.Borough").alias("borough_origen"),
        F.col("origen.Zone").alias("zona_origen"),

        F.col("destino.Borough").alias("borough_destino"),
        F.col("destino.Zone").alias("zona_destino")
    )
)

Comprueba que la integración no haya multiplicado filas y revisa nombres de zonas y distritos.

In [0]:
print("Registros integrados:", df_viajes_integrados.count())

df_viajes_integrados.select(
    "PULocationID",
    "zona_origen",
    "borough_origen",
    "DOLocationID",
    "zona_destino",
    "borough_destino"
).show(10, truncate=False)

Registros integrados: 10876240
+------------+----------------------------+--------------+------------+----------------------------+---------------+
|PULocationID|zona_origen                 |borough_origen|DOLocationID|zona_destino                |borough_destino|
+------------+----------------------------+--------------+------------+----------------------------+---------------+
|140         |Lenox Hill East             |Manhattan     |236         |Upper East Side North       |Manhattan      |
|140         |Lenox Hill East             |Manhattan     |262         |Yorkville East              |Manhattan      |
|161         |Midtown Center              |Manhattan     |68          |East Chelsea                |Manhattan      |
|231         |TriBeCa/Civic Center        |Manhattan     |13          |Battery Park City           |Manhattan      |
|163         |Midtown North               |Manhattan     |236         |Upper East Side North       |Manhattan      |
|166         |Morningside Heights

Registra una vista temporal `viajes_integrados` para validar el resultado con Spark SQL.

In [0]:
df_viajes_integrados.createOrReplaceTempView("viajes_integrados")

Cuenta viajes sin correspondencia geográfica. **Referencia:** cero zonas de origen/destino faltantes. Esto no excluye categorías genéricas como `Unknown`.

In [0]:
spark.sql("""
    SELECT
        COUNT(*) AS total_viajes,

        SUM(CASE WHEN zona_origen IS NULL THEN 1 ELSE 0 END)
            AS sin_zona_origen,

        SUM(CASE WHEN zona_destino IS NULL THEN 1 ELSE 0 END)
            AS sin_zona_destino,

        SUM(
            CASE
                WHEN zona_origen IS NULL
                  OR zona_destino IS NULL
                THEN 1 ELSE 0
            END
        ) AS sin_alguna_zona

    FROM viajes_integrados
""").show()

+------------+---------------+----------------+---------------+
|total_viajes|sin_zona_origen|sin_zona_destino|sin_alguna_zona|
+------------+---------------+----------------+---------------+
|    10876240|              0|               0|              0|
+------------+---------------+----------------+---------------+



## Etapa 7 · Persistencia con Delta Lake

Guarda una tabla Delta administrada en `clean`. Delta utiliza archivos Parquet y un registro de transacciones; `overwrite` reemplaza el contenido previo.

In [0]:
(
    df_viajes_integrados
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi_analytics.clean.viajes_integrados")
)

print("Tabla Delta creada correctamente.")

Tabla Delta creada correctamente.


Consulta la tabla persistente y valida conteo, días y fechas. **Referencia:** 10,876,240 viajes en 90 días (2025-01-01 a 2025-03-31).

In [0]:
spark.sql("""
    SELECT
        COUNT(*) AS total_viajes,
        COUNT(DISTINCT fecha_viaje) AS dias_analizados,
        MIN(fecha_viaje) AS primera_fecha,
        MAX(fecha_viaje) AS ultima_fecha
    FROM nyc_taxi_analytics.clean.viajes_integrados
""").show()

+------------+---------------+-------------+------------+
|total_viajes|dias_analizados|primera_fecha|ultima_fecha|
+------------+---------------+-------------+------------+
|    10876240|             90|   2025-01-01|  2025-03-31|
+------------+---------------+-------------+------------+



Resume las comprobaciones de la capa limpia y explica por qué la tabla persistente es la fuente del datamart.

## Resultado de la etapa clean

|Validación|Resultado|
|---|---|
|Tabla|nyc_taxi_analytics.clean.viajes_integrados|
|Registros|10,876,240|
|Días analizados|90|
|Primera fecha|01/01/2025|
|Última fecha|31/03/2025|
|Viajes sin zona de origen|0|
|Viajes sin zona de destino|0|

Con esto confirmamos que la tabla conserva los viajes válidos para nuestro análisis de movilidad y que todos tienen correspondencia con el catálogo de zonas.

## Etapa 8 · Datamart y análisis de demanda

Crea `datamart.demanda_diaria` con SQL CTAS: una fila por fecha y el total de viajes. **Referencia:** 90 filas.

In [0]:
# Crear el indicador de la demanda diaria

spark.sql("""
    CREATE OR REPLACE TABLE
        nyc_taxi_analytics.datamart.demanda_diaria
    USING DELTA
    AS
    SELECT
        fecha_viaje,
        YEAR(fecha_viaje) AS anio,
        MONTH(fecha_viaje) AS mes,
        DAYOFWEEK(fecha_viaje) AS dia_semana,
        COUNT(*) AS total_viajes
    FROM nyc_taxi_analytics.clean.viajes_integrados
    GROUP BY fecha_viaje
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

Agrupa la demanda diaria por mes y calcula el promedio por día; evita confundir totales mensuales con promedios diarios.

In [0]:
# Validar el indicador de la demanda diaria

spark.sql("""
    SELECT
        mes,
        COUNT(*) AS dias,
        SUM(total_viajes) AS viajes_mes,
        ROUND(AVG(total_viajes), 0) AS promedio_diario
    FROM nyc_taxi_analytics.datamart.demanda_diaria
    GROUP BY mes
    ORDER BY mes
""").show()

+---+----+----------+---------------+
|mes|dias|viajes_mes|promedio_diario|
+---+----+----------+---------------+
|  1|  31|   3382982|       109128.0|
|  2|  28|   3473273|       124045.0|
|  3|  31|   4019985|       129677.0|
+---+----+----------+---------------+



Interpreta la tendencia enero–marzo y sus límites: la asociación temporal no demuestra causas.

Podemos identificar tres hallazgos:
1. Marzo presentó la mayor demanda, con más de cuatro millones de viajes.
2. Entre enero y marzo, el número total de viajes aumentó aproximadamente un 18.8 %.
3. El promedio diario también creció, de 109,128 a 129,677 viajes, un incremento aproximado del 18.8 %.
El tercer punto es importante porque febrero tiene menos días. Comparar únicamente los totales mensuales podría llevarnos a conclusiones equivocadas sobre la demanda.

Compara los días de la semana con el promedio de viajes diarios. El promedio evita sesgos por días que ocurren 12 o 13 veces.

In [0]:
# Siguiente indicador: comportamiento de la demanda por día de la semana

spark.sql("""
    SELECT
        dia_semana,

        CASE dia_semana
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Lunes'
            WHEN 3 THEN 'Martes'
            WHEN 4 THEN 'Miércoles'
            WHEN 5 THEN 'Jueves'
            WHEN 6 THEN 'Viernes'
            WHEN 7 THEN 'Sábado'
        END AS nombre_dia,

        COUNT(*) AS numero_dias,

        SUM(total_viajes) AS total_viajes,

        ROUND(AVG(total_viajes), 0) AS promedio_diario

    FROM nyc_taxi_analytics.datamart.demanda_diaria

    GROUP BY dia_semana
    ORDER BY dia_semana
""").show()

+----------+----------+-----------+------------+---------------+
|dia_semana|nombre_dia|numero_dias|total_viajes|promedio_diario|
+----------+----------+-----------+------------+---------------+
|         1|   Domingo|         13|     1472542|       113272.0|
|         2|     Lunes|         13|     1281131|        98549.0|
|         3|    Martes|         12|     1384661|       115388.0|
|         4| Miércoles|         13|     1579124|       121471.0|
|         5|    Jueves|         13|     1703298|       131023.0|
|         6|   Viernes|         13|     1648604|       126816.0|
|         7|    Sábado|         13|     1806880|       138991.0|
+----------+----------+-----------+------------+---------------+



Interpreta el patrón semanal y formula hipótesis para contrastar posteriormente.

Interpretación de la demanda semanal
Los hallazgos más relevantes son:
- Sábado: es el día con mayor demanda, con 138,991 viajes diarios en promedio.
- Jueves: ocupa el segundo lugar, con 131,023 viajes.
- Lunes: registra la menor demanda, con 98,549 viajes.
- La demanda promedio de los sábados es aproximadamente 41 % mayor que la de los lunes.
Esto podría estar relacionado con actividades recreativas, comerciales y turísticas, aunque necesitaríamos información adicional para confirmar esas causas.

También podemos observar que el viernes no supera al jueves, algo que sería interesante investigar posteriormente por horarios.

Crea `demanda_horaria` con una fila por combinación día de semana/hora (máximo 168); después identifica las horas con más viajes.

In [0]:
# Siguiente indicador: ¿a qué horas se utilizan más los taxis?

spark.sql("""
    CREATE OR REPLACE TABLE
        nyc_taxi_analytics.datamart.demanda_horaria
    USING DELTA
    AS
    SELECT
        dia_semana,
        hora,
        COUNT(*) AS total_viajes

    FROM nyc_taxi_analytics.clean.viajes_integrados

    GROUP BY
        dia_semana,
        hora
""")

spark.sql("""
    SELECT
        hora,
        SUM(total_viajes) AS total_viajes

    FROM nyc_taxi_analytics.datamart.demanda_horaria

    GROUP BY hora
    ORDER BY total_viajes DESC
""").show(24)

+----+------------+
|hora|total_viajes|
+----+------------+
|  18|      773282|
|  17|      722786|
|  19|      692866|
|  21|      687339|
|  20|      675381|
|  15|      638148|
|  16|      636401|
|  22|      627269|
|  14|      606587|
|  13|      565263|
|  12|      535211|
|  11|      489563|
|  23|      474521|
|  10|      457595|
|   9|      445865|
|   8|      433546|
|   0|      331080|
|   7|      312364|
|   1|      219563|
|   6|      159331|
|   2|      141175|
|   3|       98578|
|   5|       78505|
|   4|       74021|
+----+------------+



Resume las franjas horarias de mayor y menor demanda.

Podemos distinguir tres comportamientos:
- Madrugada (00:00–05:59): demanda reducida, con su mínimo a las 04:00, cuando se registraron 74,021 viajes.
- Mañana y tarde (06:00–16:59): crecimiento progresivo, aunque con una ligera disminución a las 16:00.
- Tarde-noche (17:00–22:59): periodo de mayor actividad, con un máximo a las 18:00 de 773,282 viajes.
La demanda de las 18:00 es aproximadamente 10.4 veces la registrada a las 04:00.

Esto nos permite plantear una hipótesis: la demanda está asociada con horarios de salida del trabajo, actividades comerciales y recreativas. Sin embargo, debemos recordar que aún no hemos distinguido entre días laborales y fines de semana.

Compara combinaciones día/hora usando promedio por ocurrencia. La fórmula del notebook original usa 12 martes y 13 de los otros días para este trimestre.

In [0]:
# Siguiente indicador: combinar día de la semana y hora

spark.sql("""
    SELECT
        CASE dia_semana
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Lunes'
            WHEN 3 THEN 'Martes'
            WHEN 4 THEN 'Miércoles'
            WHEN 5 THEN 'Jueves'
            WHEN 6 THEN 'Viernes'
            WHEN 7 THEN 'Sábado'
        END AS nombre_dia,

        hora,

        total_viajes,

        ROUND(
            total_viajes /
            CASE dia_semana
                WHEN 3 THEN 12
                ELSE 13
            END, 0
        ) AS promedio_por_dia

    FROM nyc_taxi_analytics.datamart.demanda_horaria

    ORDER BY promedio_por_dia DESC
    LIMIT 15
""").show(truncate=False)

# ¿Por qué dividimos entre 12 o 13?

# Porque en nuestro trimestre tenemos 12 martes y 13 ocurrencias de cada uno de los demás días de la semana.

# Si comparamos solamente los viajes acumulados, estaríamos favoreciendo a los días que aparecen más veces.
# El promedio nos permite comparar las combinaciones en igualdad de condiciones.

# Por ejemplo, si los sábados a las 18:00 registraron 130,000 viajes durante 13 sábados, el promedio sería de 10,000 viajes por sábado en esa hora.

+----------+----+------------+----------------+
|nombre_dia|hora|total_viajes|promedio_por_dia|
+----------+----+------------+----------------+
|Sábado    |19  |131002      |10077.0         |
|Jueves    |18  |126621      |9740.0          |
|Jueves    |21  |126391      |9722.0          |
|Sábado    |18  |126210      |9708.0          |
|Sábado    |23  |125630      |9664.0          |
|Sábado    |22  |120930      |9302.0          |
|Jueves    |20  |119294      |9176.0          |
|Viernes   |18  |118979      |9152.0          |
|Jueves    |17  |116324      |8948.0          |
|Miércoles |18  |116238      |8941.0          |
|Miércoles |21  |114116      |8778.0          |
|Jueves    |19  |113134      |8703.0          |
|Jueves    |22  |113145      |8703.0          |
|Sábado    |17  |111920      |8609.0          |
|Viernes   |19  |111726      |8594.0          |
+----------+----+------------+----------------+



## Etapa 9 · Preparación de datos para Tableau Public

Consulta y presenta los 90 registros de demanda diaria, listos para descargarse como CSV desde el resultado de Databricks.

In [0]:
df_demanda_diaria = spark.sql("""
    SELECT
        fecha_viaje,
        anio,
        mes,
        dia_semana,
        total_viajes
    FROM nyc_taxi_analytics.datamart.demanda_diaria
    ORDER BY fecha_viaje
""")

display(df_demanda_diaria)

fecha_viaje,anio,mes,dia_semana,total_viajes
2025-01-01,2025,1,4,88616
2025-01-02,2025,1,5,83217
2025-01-03,2025,1,6,89494
2025-01-04,2025,1,7,96565
2025-01-05,2025,1,1,78340
2025-01-06,2025,1,2,78574
2025-01-07,2025,1,3,98369
2025-01-08,2025,1,4,110848
2025-01-09,2025,1,5,115821
2025-01-10,2025,1,6,108280


Consulta los 168 registros de demanda horaria para su exportación y posterior mapa de calor.

In [0]:
df_demanda_horaria = spark.sql("""
    SELECT
        dia_semana,
        hora,
        total_viajes
    FROM nyc_taxi_analytics.datamart.demanda_horaria
    ORDER BY dia_semana, hora
""")

display(df_demanda_horaria)

dia_semana,hora,total_viajes
1,0,108927
1,1,84827
1,2,55648
1,3,41792
1,4,25572
1,5,12415
1,6,15132
1,7,19566
1,8,28158
1,9,43885


Crea la tabla Delta agregada por zona de origen y distrito. La granularidad es una fila por zona de origen.

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE
        nyc_taxi_analytics.datamart.demanda_por_zona
    USING DELTA
    AS
    SELECT
        PULocationID AS id_zona,
        zona_origen AS zona,
        borough_origen AS distrito,
        COUNT(*) AS total_viajes
    FROM nyc_taxi_analytics.clean.viajes_integrados
    GROUP BY
        PULocationID,
        zona_origen,
        borough_origen
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

Consulta las diez zonas con más viajes iniciados. **Referencia:** Midtown Center encabeza el ranking.

In [0]:
spark.sql("""
    SELECT
        zona,
        distrito,
        total_viajes
    FROM nyc_taxi_analytics.datamart.demanda_por_zona
    ORDER BY total_viajes DESC
    LIMIT 10
""").show(truncate=False)

+----------------------------+---------+------------+
|zona                        |distrito |total_viajes|
+----------------------------+---------+------------+
|Midtown Center              |Manhattan|502500      |
|Upper East Side South       |Manhattan|483531      |
|Upper East Side North       |Manhattan|447566      |
|JFK Airport                 |Queens   |423124      |
|Times Sq/Theatre District   |Manhattan|370036      |
|Penn Station/Madison Sq West|Manhattan|359630      |
|Midtown East                |Manhattan|353033      |
|Lincoln Square East         |Manhattan|326438      |
|Union Sq                    |Manhattan|305012      |
|Murray Hill                 |Manhattan|294693      |
+----------------------------+---------+------------+



Muestra todas las zonas agregadas para exportarlas y aplicar un Top 10 dinámico en Tableau.

In [0]:
df_demanda_zonas = spark.sql("""
    SELECT
        zona,
        distrito,
        total_viajes
    FROM nyc_taxi_analytics.datamart.demanda_por_zona
    ORDER BY total_viajes DESC
""")

display(df_demanda_zonas)

zona,distrito,total_viajes
Midtown Center,Manhattan,502500
Upper East Side South,Manhattan,483531
Upper East Side North,Manhattan,447566
JFK Airport,Queens,423124
Times Sq/Theatre District,Manhattan,370036
Penn Station/Madison Sq West,Manhattan,359630
Midtown East,Manhattan,353033
Lincoln Square East,Manhattan,326438
Union Sq,Manhattan,305012
Murray Hill,Manhattan,294693


## Etapa 10 · Construcción del dashboard en Tableau Public

**Exporta únicamente los resultados agregados**, no los 10.8 millones de viajes. Desde cada `display(...)` en Databricks utiliza la descarga CSV del resultado (si la interfaz no lo permite, exporta los DataFrames agregados a un volumen).

1. **Evolución diaria:** `demanda_diaria.csv`. Columnas: `fecha_viaje` como fecha continua; Filas: `SUM(total_viajes)`; Marcas: línea.
2. **Demanda por día:** misma fuente diaria. Columnas: `dia_semana` discreto; Filas: `AVG(total_viajes)`; Marcas: barras. Crea alias de domingo a sábado.
3. **Mapa de calor:** `demanda_horaria.csv`. Columnas: `hora` discreta; Filas: `dia_semana` discreto; Marcas: cuadrado; Color: promedio de viajes por día y hora. Para este trimestre, divide por 12 los martes y por 13 los demás días, o utiliza el número de ocurrencias obtenido desde `demanda_diaria`.
4. **Top 10 zonas:** `demanda_por_zona.csv`. Filas: `zona`; Columnas: `SUM(total_viajes)`; barras horizontales; filtro Top 10 por suma de viajes.

**Dashboard:** crea un dashboard nuevo en el **mismo libro**; coloca la evolución diaria arriba, barras semanales y ranking al centro, y mapa de calor abajo. Añade título, fuente, periodo y notas metodológicas. Las fuentes agregadas son diferentes: los filtros entre hojas no funcionan automáticamente sin relaciones/acciones configuradas.

### Entregables
- Notebook ejecutado con resultados y respuestas a las preguntas.
- Captura de las validaciones principales y explicación de las reglas de limpieza.
- Cuatro hojas y un dashboard publicado en Tableau Public.
- Conclusiones: tres hallazgos sustentados por datos y dos limitaciones del análisis.

### Reflexión final
¿Por qué la ingesta se realizó en Databricks y no en la laptop? ¿Qué diferencia hay entre una vista temporal y una tabla Delta? ¿Por qué no eliminamos automáticamente tarifas negativas ni valores superiores al P99? ¿Por qué usamos `AVG(total_viajes)` para comparar días de la semana?